# Karachi AQI Prediction & Analysis

This notebook demonstrates:
1. Loading Karachi Air Quality Index (AQI) historical data.
2. Data handling: identifying missing values and detecting outliers.
3. Exploratory Data Analysis (EDA) and visualizations (missing values, distributions, correlations, box plots).
4. Training machine learning models (Random Forest Regressors) for 1-Day, 2-Day, and 3-Day lead AQI forecasts.
5. Detailed model evaluation: R2 Score, MAE, RMSE, MAPE, tolerance accuracy ($\pm 15\%$), and AQI category classification accuracy.
6. Feature importance analysis.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import os

# Set plotting style
sns.set_theme(style="whitegrid")
plt.rcParams['figure.figsize'] = (12, 6)

## 1. Load Data
We load the historical daily dataset compiled from the AQICN API and Open-Meteo Weather API.

In [ ]:
csv_path = 'aqi_historical_data.csv'
if os.path.exists(csv_path):
    df = pd.read_csv(csv_path)
    print(f"Successfully loaded {len(df)} daily observations.")
else:
    print(f"Error: {csv_path} not found! Run the backfill script first.")

In [ ]:
# Inspect dataset structure
df.info()

In [ ]:
df.head()

## 2. Data Handling: Missing Values & Outliers

### 2.1. Handling Missing Values
We check if there are any missing values in our dataset and visualize them.

In [ ]:
missing_values = df.isnull().sum()
print("Missing values per column:")
print(missing_values[missing_values > 0] if missing_values.sum() > 0 else "No missing values found!")

# Visualizing missingness
plt.figure(figsize=(10, 4))
sns.heatmap(df.isnull(), cbar=False, yticklabels=False, cmap='viridis')
plt.title("Missing Values Heatmap")
plt.show()

### 2.2. Outlier Detection
We use the Interquartile Range (IQR) method to detect outliers in major columns: AQI, PM2.5, and Temperature. Outliers in air quality data are common during dust storms, seasonal stubble burning, or weather anomalies. We will visualize them using boxplots.

In [ ]:
def detect_outliers_iqr(data, column):
    q1 = data[column].quantile(0.25)
    q3 = data[column].quantile(0.75)
    iqr = q3 - q1
    lower_bound = q1 - 1.5 * iqr
    upper_bound = q3 + 1.5 * iqr
    
    outliers = data[(data[column] < lower_bound) | (data[column] > upper_bound)]
    print(f"Column '{column}': Found {len(outliers)} outliers (bounds: [{lower_bound:.2f}, {upper_bound:.2f}])")
    return outliers, lower_bound, upper_bound

# Detect outliers in key features
outliers_aqi, lower_aqi, upper_aqi = detect_outliers_iqr(df, 'aqi')
outliers_pm25, lower_pm25, upper_pm25 = detect_outliers_iqr(df, 'pm2_5')
outliers_temp, lower_temp, upper_temp = detect_outliers_iqr(df, 'temp')

In [ ]:
# Visualizing Outliers using Box Plots
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
sns.boxplot(ax=axes[0], y=df['aqi'], color='#e74c3c')
axes[0].set_title('AQI Box Plot')
axes[0].set_ylabel('AQI Value')

sns.boxplot(ax=axes[1], y=df['pm2_5'], color='#f39c12')
axes[1].set_title('PM2.5 Box Plot')
axes[1].set_ylabel('PM2.5 Value')

sns.boxplot(ax=axes[2], y=df['temp'], color='#3498db')
axes[2].set_title('Temperature Box Plot')
axes[2].set_ylabel('Temp (°C)')

plt.suptitle('Outlier Identification in Key Variables', fontsize=16)
plt.tight_layout()
plt.show()

### 2.3. Handling Outliers (Discussion / Imputation Option)
We can cap the outliers to the upper/lower bounds of the IQR to prevent extreme anomalies from skewing our predictions, or we can keep them because extreme air quality events are real phenomena. Here, we provide a function to optionally cap the values if desired.

In [ ]:
# Optional outlier capping function
def cap_outliers(data, column, lower_bound, upper_bound):
    data_capped = data.copy()
    data_capped[column] = np.clip(data_capped[column], lower_bound, upper_bound)
    return data_capped

# For our training, we will use the original data as real AQI spikes are important to forecast,
# but you can use df_capped if you wish to train on smoothed data.
df_capped = cap_outliers(df, 'aqi', lower_aqi, upper_aqi)
df_capped = cap_outliers(df_capped, 'pm2_5', lower_pm25, upper_pm25)
print("Outliers capped in df_capped (Optional capped dataset created).")

## 3. Exploratory Data Analysis (EDA) & Visualizations

In [ ]:
# Plotting the historical AQI timeline
df['date'] = pd.to_datetime(df['date'])
plt.figure(figsize=(14, 5))
plt.plot(df['date'], df['aqi'], label='Daily AQI', color='#e74c3c', alpha=0.8)
plt.axhline(100, color='orange', linestyle='--', label='Unhealthy for Sensitive Groups (100)')
plt.axhline(150, color='red', linestyle='--', label='Unhealthy (150)')
plt.title('Karachi Daily AQI Historical Trend', fontsize=14)
plt.xlabel('Date')
plt.ylabel('AQI Value')
plt.grid(True, linestyle='--', alpha=0.6)
plt.legend()
plt.show()

In [ ]:
# Distribution of AQI
plt.figure(figsize=(10, 5))
sns.histplot(df['aqi'], kde=True, color='#2c3e50', bins=30)
plt.axvline(df['aqi'].mean(), color='red', linestyle='-', label=f"Mean AQI: {df['aqi'].mean():.2f}")
plt.axvline(df['aqi'].median(), color='blue', linestyle='--', label=f"Median AQI: {df['aqi'].median():.2f}")
plt.title('Distribution of Karachi Daily AQI', fontsize=14)
plt.xlabel('AQI Value')
plt.ylabel('Frequency')
plt.legend()
plt.show()

In [ ]:
# Correlation Heatmap of features
corr_cols = ['aqi', 'pm2_5', 'pm10', 'nitrogen_dioxide', 'sulphur_dioxide', 'carbon_monoxide', 'ozone', 'temp', 'humidity', 'wind_speed']
plt.figure(figsize=(12, 8))
sns.heatmap(df[corr_cols].corr(), annot=True, cmap='coolwarm', fmt='.2f', linewidths=0.5)
plt.title('Feature Correlation Matrix', fontsize=14)
plt.show()

## 4. Feature Engineering & Target Preparation
We create forecasting targets: `target_aqi_1d`, `target_aqi_2d`, and `target_aqi_3d` by shifting the `aqi` column back by 1, 2, and 3 days.

In [ ]:
# Targets: AQI in next 1, 2, and 3 days
df["target_aqi_1d"] = df["aqi"].shift(-1)
df["target_aqi_2d"] = df["aqi"].shift(-2)
df["target_aqi_3d"] = df["aqi"].shift(-3)

# Clean rows with missing values in target columns
df_clean = df.dropna(subset=['target_aqi_1d', 'target_aqi_2d', 'target_aqi_3d']).copy()

# Features matching the active pipeline
feature_cols = [
    "pm2_5", "pm10", "nitrogen_dioxide", "sulphur_dioxide", "carbon_monoxide", "ozone",
    "temp", "humidity", "wind_speed", "month", "day", "day_of_week",
    "pm2_5_lag_1d", "pm2_5_lag_2d", "aqi_lag_1d"
]

X = df_clean[feature_cols]
y_1d = df_clean['target_aqi_1d']
y_2d = df_clean['target_aqi_2d']
y_3d = df_clean['target_aqi_3d']

print(f"Features shape: {X.shape}")
print(f"Target shapes: 1d={y_1d.shape}, 2d={y_2d.shape}, 3d={y_3d.shape}")

## 5. Model Training & Comprehensive Evaluation
We use **Random Forest Regressors** for multi-lead predictions. To avoid data leakage, we perform a chronological train-test split (using the last 30 days of data as our validation/testing set).

We define helper functions to compute regression metrics (R2, MAE, RMSE, MAPE) and classification-like accuracy (percentage within a tolerance limit, and category-matching accuracy based on US EPA standards).

In [ ]:
def get_epa_category(aqi_val):
    """US EPA AQI Category breakdown"""
    if aqi_val <= 50:
        return 'Good'
    elif aqi_val <= 100:
        return 'Moderate'
    elif aqi_val <= 150:
        return 'Unhealthy for Sensitive'
    elif aqi_val <= 200:
        return 'Unhealthy'
    elif aqi_val <= 300:
        return 'Very Unhealthy'
    else:
        return 'Hazardous'

def compute_metrics(y_true, y_pred):
    mae = mean_absolute_error(y_true, y_pred)
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    r2 = r2_score(y_true, y_pred)
    
    # Mean Absolute Percentage Error (MAPE)
    mape = np.mean(np.abs((y_true - y_pred) / y_true)) * 100
    
    # Custom Prediction Tolerance Accuracy (percentage of predictions within +- 15% of actual)
    tolerance = 0.15
    within_tolerance = np.abs(y_true - y_pred) <= (tolerance * y_true)
    tolerance_accuracy = np.mean(within_tolerance) * 100
    
    # Category Accuracy
    cats_true = [get_epa_category(val) for val in y_true]
    cats_pred = [get_epa_category(val) for val in y_pred]
    category_accuracy = np.mean([t == p for t, p in zip(cats_true, cats_pred)]) * 100
    
    return {
        "MAE": mae,
        "RMSE": rmse,
        "R2": r2,
        "MAPE": mape,
        "Tolerance Accuracy (±15%)": tolerance_accuracy,
        "AQI Category Accuracy": category_accuracy
    }

In [ ]:
def train_and_eval_lead(X, y, label):
    # Time-series split: use last 30 days as testing set
    split_idx = len(X) - 30
    X_train, X_test = X.iloc[:split_idx], X.iloc[split_idx:]
    y_train, y_test = y.iloc[:split_idx], y.iloc[split_idx:]
    
    model = RandomForestRegressor(n_estimators=100, random_state=42, max_depth=10)
    model.fit(X_train, y_train)
    
    preds = model.predict(X_test)
    metrics = compute_metrics(y_test, preds)
    
    print(f"=== Evaluation for {label} ===")
    for k, v in metrics.items():
        if "Accuracy" in k or "MAPE" in k:
            print(f"{k}: {v:.2f}%")
        elif "R2" in k:
            print(f"{k}: {v:.3f}")
        else:
            print(f"{k}: {v:.2f}")
    print()
    
    return model, metrics, y_test, preds

print("Training lead models...")
m_1d, metrics_1d, y_test_1d, preds_1d = train_and_eval_lead(X, y_1d, "1-Day Forecast (Tomorrow)")
m_2d, metrics_2d, y_test_2d, preds_2d = train_and_eval_lead(X, y_2d, "2-Day Forecast (Day After)")
m_3d, metrics_3d, y_test_3d, preds_3d = train_and_eval_lead(X, y_3d, "3-Day Forecast (Three Days Ahead)")

In [ ]:
# Visualize Actual vs Predicted AQI for the last 30 days (1-Day Forecast)
plt.figure(figsize=(14, 6))
test_dates = df_clean['date'].iloc[-30:]
plt.plot(test_dates, y_test_1d.values, label='Actual AQI', color='#2c3e50', marker='o')
plt.plot(test_dates, preds_1d, label='Predicted AQI (1-Day Lead)', color='#e74c3c', linestyle='--', marker='x')
plt.title('Actual vs Predicted AQI (Last 30 Days Validation)', fontsize=14)
plt.xlabel('Date')
plt.ylabel('AQI Value')
plt.grid(True, linestyle='--', alpha=0.6)
plt.legend()
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

In [ ]:
# Compare Performance Metrics across lead times
metrics_names = ["R2", "MAE", "RMSE", "Tolerance Accuracy (±15%)", "AQI Category Accuracy"]
leads = ["1-Day Lead", "2-Day Lead", "3-Day Lead"]

comparison_df = pd.DataFrame({
    "1-Day Lead": [metrics_1d[m] for m in metrics_names],
    "2-Day Lead": [metrics_2d[m] for m in metrics_names],
    "3-Day Lead": [metrics_3d[m] for m in metrics_names]
}, index=metrics_names)

print("Metrics Comparison Matrix:")
comparison_df.round(3)

## 6. Feature Importance Analysis
Let's see which feature contributes most to the 1-Day lead model forecast.

In [ ]:
importances = m_1d.feature_importances_
indices = np.argsort(importances)[::-1]

plt.figure(figsize=(12, 6))
sns.barplot(x=[feature_cols[i] for i in indices], y=importances[indices], hue=[feature_cols[i] for i in indices], legend=False, palette='mako')
plt.title('Random Forest Feature Importance - Tomorrow Forecast')
plt.xlabel('Features')
plt.ylabel('Importance Score')
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()